### Setup

In [1]:
# custom imports
import nekt
import os                           
from dotenv import load_dotenv      
from typing import Optional 
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# get Nekt data access token        
load_dotenv()
nekt.data_access_token = os.getenv("NEKT_DATA_ACCESS_TOKEN")

### Functions

In [2]:
# auxiliary functions
def extract_nekt_table(table_name: str, layer_name: str) -> DataFrame:
    return nekt.load_table(layer_name=layer_name, table_name=table_name)

def save_nekt_table(
    df: DataFrame, 
    layer_name: str, 
    table_name: str,
    folder_name: Optional[str] = None 
):
    nekt.save_table(
        df=df,
        layer_name=layer_name,
        table_name=table_name,
        folder_name=folder_name
    )

def display(df: DataFrame, limit=10):
    return df.limit(limit).toPandas()

### Extraction

In [14]:
# clickup
df_bronze_clickup_users        = extract_nekt_table("users"        , "Bronze")
df_bronze_clickup_spaces       = extract_nekt_table("spaces"       , "Bronze")
df_bronze_clickup_time_entries = extract_nekt_table("time_entries" , "Bronze")

params: {'include_layer_database_name': 'true'}
Table details: {'id': '7adcd28a-c0e3-4a6a-aea2-1eeef140cee7', 'name': 'users', 'slug': 'users', 'database_id': 'users', 'layer_database_name': 'stech_solucoes_tecnologicas_bronze', 'folder': '2a5ad9ed-e5db-46ed-b2f8-3bf559fae219', 'description': None, 's3_path': 'bq://nekt-hosted-infrastructure.stech_solucoes_tecnologicas_bronze.users', 'layer': '85675f10-b2e9-44ea-8e30-a88b0b545b89', 'deleted': False, 'created_at': '2026-02-26T11:33:03.805939-03:00', 'updated_at': '2026-03-02T15:28:42.292806-03:00'}
params: {'include_layer_database_name': 'true'}
Table details: {'id': '5169a424-7ea2-43be-98e5-f26828090499', 'name': 'spaces', 'slug': 'spaces', 'database_id': 'spaces', 'layer_database_name': 'stech_solucoes_tecnologicas_bronze', 'folder': '2a5ad9ed-e5db-46ed-b2f8-3bf559fae219', 'description': None, 's3_path': 'bq://nekt-hosted-infrastructure.stech_solucoes_tecnologicas_bronze.spaces', 'layer': '85675f10-b2e9-44ea-8e30-a88b0b545b89', 'delet

In [18]:
# conta azul
df_bronze_contaazul_categories          = extract_nekt_table("categorias"           , "Bronze")
df_bronze_contaazul_costs_centers       = extract_nekt_table("centros_de_custos"    , "Bronze")
df_bronze_contaazul_financial_accounts  = extract_nekt_table("contas_financeiras"   , "Bronze")
df_bronze_contaazul_account_payables    = extract_nekt_table("despesas"             , "Bronze")
df_bronze_contaazul_installments        = extract_nekt_table("parcelas"             , "Bronze")
# df_bronze_contaazul_people_details      = extract_nekt_table("pessoas_detalhes"     , "Bronze")
df_bronze_contaazul_people_list         = extract_nekt_table("pessoas_lista"        , "Bronze")
df_bronze_contaazul_products            = extract_nekt_table("produtos"             , "Bronze")
df_bronze_contaazul_account_receivables = extract_nekt_table("receitas"             , "Bronze")
df_bronze_contaazul_services            = extract_nekt_table("servico"              , "Bronze")
df_bronze_contaazul_sales_details       = extract_nekt_table("vendas_detalhes"      , "Bronze")
df_bronze_contaazul_sales_items         = extract_nekt_table("vendas_lista"         , "Bronze")
df_bronze_contaazul_sales_list          = extract_nekt_table("vendas_itens"         , "Bronze")
df_bronze_contaazul_dre_categories      = extract_nekt_table("categorias_dre"       , "Bronze")

params: {'include_layer_database_name': 'true'}
Table details: {'id': '6dbe0d38-d61d-4516-8087-a1d4c0fda04d', 'name': 'categorias', 'slug': 'categorias', 'database_id': 'categorias', 'layer_database_name': 'stech_solucoes_tecnologicas_bronze', 'folder': 'e638118c-942e-4cb4-8345-83d7e8a2741a', 'description': None, 's3_path': 'bq://nekt-hosted-infrastructure.stech_solucoes_tecnologicas_bronze.categorias', 'layer': '85675f10-b2e9-44ea-8e30-a88b0b545b89', 'deleted': False, 'created_at': '2026-03-09T12:04:18.509606-03:00', 'updated_at': '2026-03-16T09:45:27.311345-03:00'}
params: {'include_layer_database_name': 'true'}
Table details: {'id': '48ed97c8-1954-4e99-a2a5-7c49d72f653f', 'name': 'centros_de_custos', 'slug': 'centros_de_custos', 'database_id': 'centros_de_custos', 'layer_database_name': 'stech_solucoes_tecnologicas_bronze', 'folder': 'e638118c-942e-4cb4-8345-83d7e8a2741a', 'description': None, 's3_path': 'bq://nekt-hosted-infrastructure.stech_solucoes_tecnologicas_bronze.centros_de_

### Tranformations

In [15]:
# clickup
## users
df_silver_clickup_users = df_bronze_clickup_users.select(
      F.col("id").cast("integer")     .alias("user_id")
    , F.col("username")               .alias("user_name")
).filter(
    F.col("user_name").isNotNull()
).dropDuplicates(
    ["user_id"]
)

## spaces
df_silver_clickup_spaces = df_bronze_clickup_spaces.select(
      F.col("id").cast("long")        .alias("space_id")
    , F.col("name")                   .alias("space_name")
).filter(
    F.col("space_name").isNotNull()
).dropDuplicates(
    ["space_id"]
)

## time entries
df_silver_clickup_time_entries = df_bronze_clickup_time_entries.select(
      F.col("id").cast("long")                        .alias("interval_id")
    , F.col("wid").cast("integer")                    .alias("team_id")
    , F.col("user.id").cast("integer")                .alias("user_id")
    , F.col("user.username")                          .alias("user_name")
    , F.col("task_location.space_id").cast("long")    .alias("space_id")
    , F.col("task.id").alias("task_id")
    , F.col("start").cast("long")                     .alias("interval_date_start_ms")
    , F.to_timestamp(F.col("start") / 1000)           .alias("interval_date_start_iso")
    , F.col("end").cast("long")                       .alias("interval_date_end_ms")
    , F.to_timestamp(F.col("end") / 1000)             .alias("interval_date_end_iso")
    , F.col("at").cast("long")                        .alias("interval_date_added_ms")
    , F.to_timestamp(F.col("at") / 1000)              .alias("interval_date_added_iso")
).filter(
      F.col("interval_id").isNotNull()
    & F.col("user_id").isNotNull()
    & F.col("interval_date_start_ms").isNotNull()
    & F.col("interval_date_end_ms").isNotNull()
    & (F.col("interval_date_end_ms") > F.col("interval_date_start_ms"))  # validating business rule
).dropDuplicates(
    ["interval_id"]
)

In [26]:
# conta azul
## categories         

## costs_centers      

## financial_accounts
 
## account payables (expenses)
df_silver_contaazul_account_payables = df_bronze_contaazul_account_payables.select(
      F.col("id")
    , F.col("descricao")
    , F.col("data_vencimento")
    , F.col("status")
    , F.col("total")
    , F.col("nao_pago")
    , F.col("pago")
    , F.col("data_criacao")
    , F.col("data_alteracao")
    , F.current_timestamp().alias("loaded_at")
).filter(
      F.col("id").isNotNull()
    & F.col("data_criacao").isNotNull()
).dropDuplicates(
    ["id"]
)  

# ## installments
df_silver_contaazul_installments = df_bronze_contaazul_installments.select(
      F.col("id")                       .alias("parcela_id")
    , F.col("status")                   .alias("parcela_status")
    , F.col("evento.condicao_pagamento").alias("condicao_pagamento")
    , F.col("referencia")
    , F.col("evento.agendado")          .alias("agendado")
    , F.col("evento.tipo")              .alias("tipo_evento")
    # , F.col("")                         .alias("rateio")
    # , F.col("")                         .alias("conciliado")
    # , F.col("")                         .alias("valor_pago")
    # , F.col("")                         .alias("data_vencimento")
    # , F.col("")                         .alias("data_pagamento_previsto")
    # , F.col("")                         .alias("descricao")
    # , F.col("")                         .alias("id_conta_financeira")
    # , F.col("")                         .alias("metodo_pagamento")
    # , F.col("")                         .alias("parent_evento_id")
    # , F.col("")                         .alias("rateio_id_categoria")
    # , F.col("")                         .alias("rateio_nome_categoria")
    # , F.col("")                         .alias("rateio_valor")
    # , F.col("")                         .alias("rateio_centro_custo_id")
    # , F.col("")                         .alias("rateio_centro_custo_valor")
    , F.current_timestamp()             .alias("loaded_at")    
    , F.current_timestamp()             .alias("parcela_loaded_at")    
).dropDuplicates(
    ["parcela_id"]
)  

## settled installments
df_silver_contaazul_settled_installments = df_bronze_contaazul_installments.select(
      F.col("id")           .alias("parcela_id")
    , F.explode("baixas")   .alias("baixa")
).select(
      F.col("parcela_id")
    , F.col("baixa.id")                             .alias("baixa_id")
    , F.col("baixa.versao")                         .alias("baixa_versao")
    , F.col("baixa.data_pagamento")                 .alias("baixa_data_pagamento") 
    , F.col("baixa.id_reconciliacao")               .alias("baixa_id_reconciliacao")
    , F.col("baixa.id_solicitacao_cobranca")        .alias("baixa_solicitacao_cobranca")
    , F.col("baixa.observacao")                     .alias("baixa_observacao")
    , F.col("baixa.metodo_pagamento")               .alias("baixa_metodo_pagamento")
    , F.col("baixa.origem")                         .alias("baixa_origem")
    , F.col("baixa.id_recibo_digital")              .alias("baixa_id_recibo_digital")
    , F.col("baixa.tipo_evento_financeiro")         .alias("baixa_tipo_evento_financeiro")
    , F.col("baixa.nsu")                            .alias("baixa_nsu")
    , F.col("baixa.id_referencia")                  .alias("baixa_id_referencia")
    , F.col("baixa.atualizado_em")                  .alias("baixa_atualizado_em")
    , F.col("baixa.valor_composicao.desconto")      .alias("baixa_desconto")
    , F.col("baixa.valor_composicao.juros")         .alias("baixa_juros")
    , F.col("baixa.valor_composicao.multa")         .alias("baixa_multa")
    , F.col("baixa.valor_composicao.taxa")          .alias("baixa_taxa")
    , F.col("baixa.valor_composicao.valor_bruto")   .alias("baixa_valor_bruto")
    , F.col("baixa.valor_composicao.valor_liquido") .alias("baixa_valor_liquido")
    , F.current_timestamp()                         .alias("loaded_at")     
).filter(
      F.col("baixa.id").isNotNull()
).dropDuplicates(
    ["parcela_id", "baixa_id"]
)
## people_list        

## products           

## account receivables (revenues)
df_silver_contaazul_account_receivables = df_bronze_contaazul_account_receivables.select(
      F.col("id")
    , F.col("descricao")
    , F.col("data_vencimento")
    , F.col("status")
    , F.col("total")
    , F.col("nao_pago")
    , F.col("pago")
    , F.col("data_criacao")
    , F.col("data_alteracao")
    # , F.col("categorias.id")  .alias("categoria_principal_id")
    # , F.col("categorias.nome").alias("categoria_principal_nome")
    # , F.col("cliente.id")     .alias("cliente_id")
    # , F.col("cliente.nome")   .alias("cliente_nome")
    , F.current_timestamp().alias("loaded_at")
).filter(
      F.col("id").isNotNull()
    & F.col("data_criacao").isNotNull()
).dropDuplicates(
    ["id"]
)

## services           

## sales_details      

## sales_items        

## sales_list         

## dre_categories


### Saving

In [ ]:
# clickup
# save_nekt_table(df_silver_users         , "Silver", "users",        folder_name="studio_61_clickup")
# save_nekt_table(df_silver_spaces        , "Silver", "spaces",       folder_name="studio_61_clickup")
# save_nekt_table(df_silver_time_entries  , "Silver", "time_entries", folder_name="studio_61_clickup")

In [ ]:
# conta azul
# save_nekt_table(df_silver_users         , "Silver", "users",        folder_name="studio_61_clickup")
# save_nekt_table(df_silver_spaces        , "Silver", "spaces",       folder_name="studio_61_clickup")
# save_nekt_table(df_silver_time_entries  , "Silver", "time_entries", folder_name="studio_61_clickup")